# GlassBox on Gemma-2-2B-it  (Kaggle)

The full app on **google/gemma-2-2b-it** behind one public URL. Kaggle's **T4 x2** session has ~29 GB system RAM, which loads Gemma comfortably — a free Colab T4 (~13 GB) often runs out of memory during the load.

**Before you run:**
1. Right panel -> **Settings** -> **Accelerator** -> **GPU T4 x2**, and **Internet** -> **On**.
2. Accept the **gemma-2-2b-it** license with the *same* HF account as your token: https://huggingface.co/google/gemma-2-2b-it  (Gemma Scope is public — no license needed).
3. **Run All.** Paste your token in the login cell, then open the URL the launch cell prints. Switch the model selector to `gemma-2-2b-it` (first load ~1-2 min).

In [ ]:
!git clone -b main https://github.com/santoshcheethiralame-dot/GlassBox.git /kaggle/working/GlassBox
%cd /kaggle/working/GlassBox
!pip install -q sae_lens accelerate fastapi "uvicorn[standard]" scikit-learn requests

In [ ]:
from huggingface_hub import login
login(token="hf_PASTE_YOUR_TOKEN_HERE")  # or: Add-ons -> Secrets -> HF_TOKEN, then UserSecretsClient().get_secret('HF_TOKEN')

In [ ]:
!cd frontend && npm ci --no-audit --no-fund --silent && npm run build
!rm -rf backend/static && cp -r frontend/dist backend/static
print("frontend built")

In [ ]:
import os, re, time, subprocess, urllib.request, requests

if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared",
    )
    os.chmod("cloudflared", 0o755)

backend = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "7860"], cwd="backend"
)

print("starting backend (loads GPT-2; Gemma loads on first use)...")
for _ in range(90):
    try:
        requests.get("http://127.0.0.1:7860/api/health", timeout=2)
        break
    except Exception:
        time.sleep(2)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
for line in iter(tunnel.stdout.readline, ""):
    print(line, end="")
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

print("\n" + "=" * 64)
print("  GlassBox is LIVE ->", url)
print("  Switch the model selector to 'gemma-2-2b-it' (first load ~1-2 min).")
print("=" * 64)

## Optional — run the hallucination experiment on Gemma directly

In [ ]:
import requests

r = requests.post(
    "http://127.0.0.1:7860/api/experiment",
    json={"model_key": "gemma-2-2b-it", "layer": 20},
    timeout=900,
).json()

print(f"grounded {r['n_grounded']}/{r['n_total']}  ·  confabulated {r['n_confabulated']}\n")
for row in r["rows"]:
    print(f"  {row['subject']:30} {row['predicted']:14} {row['label']:13} ld={row['logit_diff']:+.2f}")
f = r["flip"]
print(f"\nFLIP: {f['subject']}  ·  ablate f/{f['feature']}  [{f['feature_label']}]")
print(f"  logit_diff (grounded - parametric): {f['ld_before']:+.2f} -> {f['ld_after']:+.2f}  (shift {f['shift']:+.2f})")